# Replace c-GC outputs for aggregate inputs

Run this after all four fast c-GC split rerun notebooks finish. It replaces only `cgc` and `cgc_star` graph artifacts in `outputs/simulation/core_1000_*`, so `simulation_benchmark_aggregate.ipynb` can run unchanged afterwards.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "ica_denoising").exists():
            return path
    raise RuntimeError("Could not find the ica-denoising repository root.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".cache" / "matplotlib"))
(REPO_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ica_denoising.simulation.graph_replacement import replace_graph_estimator_outputs

OUTPUT_DIR = REPO_ROOT / "outputs" / "simulation"
SOURCE_PREFIX = "core_1000_cgc_fast"
TARGET_PREFIX = "core_1000"
METHOD_SPLITS = ("fastica", "infomax", "jade", "sobi")
ESTIMATORS = ("cgc", "cgc_star")

print(json.dumps({
    "output_dir": str(OUTPUT_DIR.relative_to(REPO_ROOT)),
    "source_prefix": SOURCE_PREFIX,
    "target_prefix": TARGET_PREFIX,
    "method_splits": list(METHOD_SPLITS),
    "estimators": list(ESTIMATORS),
}, indent=2))

In [ ]:
def run_replacement(*, dry_run: bool) -> pd.DataFrame:
    rows = []
    for method in METHOD_SPLITS:
        report = replace_graph_estimator_outputs(
            OUTPUT_DIR / f"{SOURCE_PREFIX}_{method}",
            OUTPUT_DIR / f"{TARGET_PREFIX}_{method}",
            estimators=ESTIMATORS,
            dry_run=dry_run,
        )
        rows.append({
            "method_split": method,
            "checked": report.checked,
            "replaced": report.replaced,
            "estimators": ",".join(report.estimators),
            "source_root": Path(report.source_root).name,
            "target_root": Path(report.target_root).name,
            "dry_run": dry_run,
        })
    return pd.DataFrame(rows)


dry_run_report = run_replacement(dry_run=True)
dry_run_report

In [ ]:
replacement_report = run_replacement(dry_run=False)
replacement_report